# Setup

## Imports

In [ ]:
# matplotlib
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import MaxNLocator

# others
import airr
import colorspacious as cs
import numpy as np
import pandas as pd
import seaborn as sns
from tabulate import tabulate

# add repertoire_zoo
import repertoire_zoo.bird as br
import repertoire_zoo.giraffe as gr

## Setup directories - Read in data

In [ ]:
# Define project/job uuid
project_id = ... # project uuid
job_id = ... # jov uuid

# pull out repertoires and names
groups_of_interest = [ ... ] # insert groups to compare (to compare all replace `...` with `None`)

# identify control group(s) to compare against
control_group_names = [ ... ] # insert control groups (for no control replace `...` with `None`)

# Define data directories
data_dir = ... #/path/to/data/directory/
repcalc_dir = data_dir + 'vdjserver/'+job_id+'/'
analysis_dir = data_dir + 'analysis/'
fig_dir = analysis_dir + 'figures/'

# Read in repertoire and repertoire group info
# The default name (repertoires.airr.json) is being used, but custom airr.json files can
# be used for defining repertoires and repertoire groups. 
repertoires_airr_json_filename = 'repertoires.airr.json'
sample_info_df = br.read_repertoire_info_airr(repcalc_dir + repertoires_airr_json_filename)
repertoire_group_df = br.read_repertoire_group_info_airr(repcalc_dir + repertoires_airr_json_filename)
data = airr.read_airr(repcalc_dir + repertoires_airr_json_filename)

if groups_of_interest == [None]:
    repertoire_groups_temp = { obj['repertoire_group_id'] : obj for obj in data['RepertoireGroup'] }
else:
    repertoire_groups_temp = { obj['repertoire_group_id'] : obj for obj in data['RepertoireGroup'] if obj['repertoire_group_name'] in groups_of_interest }
repertoires = { obj['repertoire_id'] : obj for obj in data['Repertoire']}

order_map = {name: idx for idx, name in enumerate(groups_of_interest)}
repertoire_groups = dict(sorted(
    repertoire_groups_temp.items(),
    key=lambda item: order_map.get(item[1]['repertoire_group_name'], len(groups_of_interest))
))

# make groups tuple
groups_tuple = [
    (
        repertoire_groups[rep_group_id]['repertoire_group_id'],
        repertoire_groups[rep_group_id]['repertoire_group_name'],
        [reps.get('repertoire_id') for reps in repertoire_groups[rep_group_id]['repertoires']]
    ) for rep_group_id in repertoire_groups
]

for i, rep_group_id in enumerate(repertoire_groups):
    print(rep_group_id, i, repertoire_groups[rep_group_id]['repertoire_group_name'])

condition_counts = {e[1] : e[2] for e in groups_tuple}

## Colormaps

In [ ]:
# set hue order (use condition names)
# By default the hue_order will be the same as the groups of interest. 
# If this is different, redfine `hue_order` as a list of groups from `groups_of_interest`
hue_order = groups_of_interest

cmap = mpl.colormaps['nipy_spectral'] # gist_rainbow
cmin = 0.0
cmax = 1.0

# V
v_genes = br.load_gene_usage_group_data(
    repcalc_dir=repcalc_dir,
    groups=groups_tuple,
    processing_stage='igblast.makedb.gene.clone',
    call_type='v_call',
    level='gene'
)['gene'].unique()
v_genes_cmap = {
    v_gene : cmap(np.linspace(cmin,cmax,len(v_genes))[i])
    for i, v_gene in enumerate(v_genes)
}

v_subgroups = br.load_gene_usage_group_data(
    repcalc_dir=repcalc_dir,
    groups=groups_tuple,
    processing_stage='igblast.makedb.gene.clone',
    call_type='v_call',
    level='subgroup'
)['gene'].unique()
v_subgroup_cmap = {
    v_subgroup : cmap(np.linspace(cmin,cmax,len(v_subgroups))[i])
    for i, v_subgroup in enumerate(v_subgroups)
}

# J
j_genes = br.load_gene_usage_group_data(
    repcalc_dir=repcalc_dir,
    groups=groups_tuple,
    processing_stage='igblast.makedb.gene.clone',
    call_type='j_call',
    level='gene'
)['gene'].unique()
j_genes_cmap = {
    j_gene : cmap(np.linspace(cmin,cmax,len(j_genes))[i])
    for i, j_gene in enumerate(j_genes)
}

j_subgroups = br.load_gene_usage_group_data(
    repcalc_dir=repcalc_dir,
    groups=groups_tuple,
    processing_stage='igblast.makedb.gene.clone',
    call_type='j_call',
    level='subgroup'
)['gene'].unique()
j_subgroup_cmap = {
    j_subgroup : cmap(np.linspace(cmin,cmax,len(j_subgroups))[i])
    for i, j_subgroup in enumerate(j_subgroups)
}

## cond_cmap
conds = br.load_and_prepare_data_junction_aa(
    repcalc_dir=repcalc_dir,
    groups=groups_tuple,
    processing_stage='.igblast.makedb.allele.clone.group.',
    call_type='junction_aa_length'
)['condition'].unique()

# Colorblind safe colors and colorblind analysis
# number of colors = number of base groups to compare
base_colors = ['#A577F9', '#699B59', '#D36135', '#086788', '#FFBC42']
cond_cmap = {
    cond : base_colors[i]
    for i, cond in enumerate(conds)
}

names = [cond[1] for cond in groups_tuple]

deuteranomaly_simulated_cmap = cs.cspace_convert(
    [mcolors.to_rgb(e) for e in cond_cmap.values()], 
    start = {
        "name": "sRGB1+CVD",
        "cvd_type": "deuteranomaly",
        "severity": 100
    },
    end = "sRGB1"
)

protanomaly_simulated_cmap = cs.cspace_convert(
    [mcolors.to_rgb(e) for e in cond_cmap.values()], 
    start = {
        "name": "sRGB1+CVD",
        "cvd_type": "protanomaly",
        "severity": 100
    },
    end = "sRGB1"
)

tritanomaly_simulated_cmap = cs.cspace_convert(
    [mcolors.to_rgb(e) for e in cond_cmap.values()], 
    start = {
        "name": "sRGB1+CVD",
        "cvd_type": "tritanomaly",
        "severity": 100
    },
    end = "sRGB1"
)

fig, axes = plt.subplots(4, 1, figsize=(len(list(cond_cmap.keys())), 4))
colorblind_simulated_cmaps = [deuteranomaly_simulated_cmap, protanomaly_simulated_cmap, tritanomaly_simulated_cmap]
gr.show_colormap(cond_cmap, axes[0], False)
for i, (ax, colorblind_simulated_cmap) in enumerate(zip(axes[1:], colorblind_simulated_cmaps)):
    colorblind_simulated_cmap = [np.clip(e, 0, 1) for e in colorblind_simulated_cmap]
    colorblind_simulated_cmap = {k:v for k,v in zip(cond_cmap.keys(), colorblind_simulated_cmap)}
    if i != 2: gr.show_colormap(colorblind_simulated_cmap, ax, False)
    else: gr.show_colormap(colorblind_simulated_cmap, ax, True, 2.5)

axes[0].set_ylabel('Normal', rotation=0, labelpad=20)
axes[1].set_ylabel('Deuteranomaly', rotation=0, labelpad=39)
axes[2].set_ylabel('Protanomaly', rotation=0, labelpad=32)
axes[3].set_ylabel('Tritanomaly', rotation=0, labelpad=32)
fig.suptitle('Colorblind Analysis', fontsize=24, y=1.02)
fig.show()

# Replacement frequency and RS ratio in CDR and FWR

## Hedgehog of overall replacement frequency

In [ ]:
nrows = 1
ncols = 3
figsize = (10,10) # nrows and ncols should take care of overal fig size so no need to alter, 
                  # but it's possible to adjust figsize manually in plt.subplots()

fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=((figsize[0]*ncols), (figsize[1]*nrows)),
    subplot_kw={'projection':'polar'})

max_y_lim=0
for i, group in enumerate(groups_tuple):
    if type(axes) == np.ndarray:
        if i==0:
            axes = axes.flatten()
        ax = axes[i]
    else:
        ax = axes
    gr.plot_mutational_hedgehog(
        df=br.load_and_prepare_mutation_data(
            mutation_data_dir=repcalc_dir,
            repertoire_groups=[group],
            trim_codons=16,
            processing_stage='igblast.makedb.gene.clone.mutations'
        ),
        title={'t':f'{group[1]}, N={len(group[2])}', 'fontsize':16},
        ax=ax
    )
    if max_y_lim < ax.get_ylim()[1]:
        max_y_lim = ax.get_ylim()[1]


fontsize=int(max(24, 1.25 * fig.get_size_inches()[0]))
fig.suptitle(**{'t':'Mutational Replacement Frequency Diagram','y':1.02, 'fontsize':fontsize})

for ax in axes:
    ax.set_ylim(-0.15, max_y_lim)
    ax.yaxis.grid(True)
    ax.yaxis.set_major_locator(MaxNLocator(7))

    ticks = ax.get_yticks()
    ax.set_yticks([t for t in ticks if t >=0])  # remove the last one

plt.tight_layout()
fig.show()

## Dataframe setup

In [ ]:
file_path = f"{repcalc_dir}igblast.makedb.gene.clone.mutations.repertoire_group.frequency.mutational_report.csv"
# file_path = f"{repcalc_dir}igblast.makedb.gene.clone.mutations.repertoire.frequency.mutational_report.csv"
group_muts = pd.read_csv(file_path, index_col='repertoire_group_id')
group_muts = group_muts[group_muts.index.isin([gt[0] for gt in groups_tuple])]

ratio_cols = [ 'mu_ratio_fwr_avg', 'mu_ratio_fwr_std', 'mu_ratio_fwr_N', 'mu_ratio_fwr_r_avg', 'mu_ratio_fwr_r_std', 'mu_ratio_fwr_r_N', 'mu_ratio_fwr_s_avg', 'mu_ratio_fwr_s_std', 'mu_ratio_fwr_s_N', 
               'mu_ratio_cdr_avg', 'mu_ratio_cdr_std', 'mu_ratio_cdr_N', 'mu_ratio_cdr_r_avg', 'mu_ratio_cdr_r_std', 'mu_ratio_cdr_r_N', 'mu_ratio_cdr_s_avg', 'mu_ratio_cdr_s_std', 'mu_ratio_cdr_s_N', 
               'mu_rsratio_cdr_avg', 'mu_rsratio_cdr_std', 'mu_rsratio_cdr_N', 'mu_rsratio_fwr_avg', 'mu_rsratio_fwr_std', 'mu_rsratio_fwr_N', 
               'mu_ratio_fwr_aa_avg', 'mu_ratio_fwr_aa_std', 'mu_ratio_fwr_aa_N', 'mu_ratio_fwr_r_aa_avg', 'mu_ratio_fwr_r_aa_std', 'mu_ratio_fwr_r_aa_N', 'mu_ratio_fwr_s_aa_avg', 'mu_ratio_fwr_s_aa_std', 'mu_ratio_fwr_s_aa_N',
               'mu_ratio_cdr_aa_avg', 'mu_ratio_cdr_aa_std', 'mu_ratio_cdr_aa_N', 'mu_ratio_cdr_r_aa_avg', 'mu_ratio_cdr_r_aa_std', 'mu_ratio_cdr_r_aa_N', 'mu_ratio_cdr_s_aa_avg', 'mu_ratio_cdr_s_aa_std', 'mu_ratio_cdr_s_aa_N',
               'mu_rsratio_cdr_aa_avg', 'mu_rsratio_cdr_aa_std', 'mu_rsratio_cdr_aa_N', 'mu_rsratio_fwr_aa_avg', 'mu_rsratio_fwr_aa_std', 'mu_rsratio_fwr_aa_N',]

freq_cols = [ 'mu_freq_avg', 'mu_freq_std', 'mu_freq_N', 'mu_freq_r_avg', 'mu_freq_r_std', 'mu_freq_r_N', 'mu_freq_s_avg', 'mu_freq_s_std', 'mu_freq_s_N',
              'mu_freq_aa_avg', 'mu_freq_aa_std', 'mu_freq_aa_N', 'mu_freq_r_aa_avg', 'mu_freq_r_aa_std', 'mu_freq_r_aa_N', 'mu_freq_s_aa_avg', 'mu_freq_s_aa_std', 'mu_freq_s_aa_N', 
              'mu_freq_fwr_avg', 'mu_freq_fwr_std', 'mu_freq_fwr_N', 'mu_freq_fwr_r_avg', 'mu_freq_fwr_r_std', 'mu_freq_fwr_r_N', 'mu_freq_fwr_s_avg', 'mu_freq_fwr_s_std', 'mu_freq_fwr_s_N',
              'mu_freq_cdr_avg', 'mu_freq_cdr_std', 'mu_freq_cdr_N', 'mu_freq_cdr_r_avg', 'mu_freq_cdr_r_std', 'mu_freq_cdr_r_N', 'mu_freq_cdr_s_avg', 'mu_freq_cdr_s_std', 'mu_freq_cdr_s_N',
              'mu_freq_fwr1_avg', 'mu_freq_fwr1_std', 'mu_freq_fwr1_N', 'mu_freq_fwr1_r_avg', 'mu_freq_fwr1_r_std', 'mu_freq_fwr1_r_N', 'mu_freq_fwr1_s_avg', 'mu_freq_fwr1_s_std', 'mu_freq_fwr1_s_N', 
              'mu_freq_cdr1_avg', 'mu_freq_cdr1_std', 'mu_freq_cdr1_N', 'mu_freq_cdr1_r_avg', 'mu_freq_cdr1_r_std', 'mu_freq_cdr1_r_N', 'mu_freq_cdr1_s_avg', 'mu_freq_cdr1_s_std', 'mu_freq_cdr1_s_N', 
              'mu_freq_fwr2_avg', 'mu_freq_fwr2_std', 'mu_freq_fwr2_N', 'mu_freq_fwr2_r_avg', 'mu_freq_fwr2_r_std', 'mu_freq_fwr2_r_N', 'mu_freq_fwr2_s_avg', 'mu_freq_fwr2_s_std', 'mu_freq_fwr2_s_N',
              'mu_freq_cdr2_avg', 'mu_freq_cdr2_std', 'mu_freq_cdr2_N', 'mu_freq_cdr2_r_avg', 'mu_freq_cdr2_r_std', 'mu_freq_cdr2_r_N', 'mu_freq_cdr2_s_avg', 'mu_freq_cdr2_s_std', 'mu_freq_cdr2_s_N', 
              'mu_freq_fwr3_avg', 'mu_freq_fwr3_std', 'mu_freq_fwr3_N', 'mu_freq_fwr3_r_avg', 'mu_freq_fwr3_r_std', 'mu_freq_fwr3_r_N', 'mu_freq_fwr3_s_avg', 'mu_freq_fwr3_s_std', 'mu_freq_fwr3_s_N', 
              'mu_freq_fwr_aa_avg', 'mu_freq_fwr_aa_std', 'mu_freq_fwr_aa_N', 'mu_freq_fwr_r_aa_avg', 'mu_freq_fwr_r_aa_std', 'mu_freq_fwr_r_aa_N', 'mu_freq_fwr_s_aa_avg', 'mu_freq_fwr_s_aa_std',  'mu_freq_fwr_s_aa_N', 
              'mu_freq_cdr_aa_avg', 'mu_freq_cdr_aa_std', 'mu_freq_cdr_aa_N', 'mu_freq_cdr_r_aa_avg', 'mu_freq_cdr_r_aa_std', 'mu_freq_cdr_r_aa_N', 'mu_freq_cdr_s_aa_avg', 'mu_freq_cdr_s_aa_std', 'mu_freq_cdr_s_aa_N', 
              'mu_freq_fwr1_aa_avg', 'mu_freq_fwr1_aa_std', 'mu_freq_fwr1_aa_N', 'mu_freq_fwr1_r_aa_avg', 'mu_freq_fwr1_r_aa_std', 'mu_freq_fwr1_r_aa_N', 'mu_freq_fwr1_s_aa_avg', 'mu_freq_fwr1_s_aa_std', 'mu_freq_fwr1_s_aa_N',
              'mu_freq_cdr1_aa_avg', 'mu_freq_cdr1_aa_std', 'mu_freq_cdr1_aa_N', 'mu_freq_cdr1_r_aa_avg', 'mu_freq_cdr1_r_aa_std', 'mu_freq_cdr1_r_aa_N', 'mu_freq_cdr1_s_aa_avg', 'mu_freq_cdr1_s_aa_std', 'mu_freq_cdr1_s_aa_N',
              'mu_freq_fwr2_aa_avg', 'mu_freq_fwr2_aa_std', 'mu_freq_fwr2_aa_N', 'mu_freq_fwr2_r_aa_avg', 'mu_freq_fwr2_r_aa_std', 'mu_freq_fwr2_r_aa_N', 'mu_freq_fwr2_s_aa_avg', 'mu_freq_fwr2_s_aa_std', 'mu_freq_fwr2_s_aa_N', 
              'mu_freq_cdr2_aa_avg', 'mu_freq_cdr2_aa_std', 'mu_freq_cdr2_aa_N', 'mu_freq_cdr2_r_aa_avg', 'mu_freq_cdr2_r_aa_std', 'mu_freq_cdr2_r_aa_N', 'mu_freq_cdr2_s_aa_avg', 'mu_freq_cdr2_s_aa_std', 'mu_freq_cdr2_s_aa_N',
              'mu_freq_fwr3_aa_avg', 'mu_freq_fwr3_aa_std', 'mu_freq_fwr3_aa_N', 'mu_freq_fwr3_r_aa_avg', 'mu_freq_fwr3_r_aa_std', 'mu_freq_fwr3_r_aa_N', 'mu_freq_fwr3_s_aa_avg', 'mu_freq_fwr3_s_aa_std', 'mu_freq_fwr3_s_aa_N'
]

ratio_df = group_muts.loc[:,ratio_cols]
ratio_df['rearrangement'] = ratio_df.index.to_series().str.extract(r'-(\d+)$')[0].fillna('All').astype(str)
# ratio_df['parent_group'] = [repertoire_groups[ratio_df.index[i]].get('repertoire_group_name') if i < 5 else ' '.join(repertoire_groups[ratio_df.index[i]].get('repertoire_group_name').split(' ')[0:-1]) for i in range(ratio_df.shape[0])]
ratio_df = ratio_df.loc[:,ratio_df.columns[-2:].to_list()+ratio_df.columns[:-2].to_list()]
ratio_df = ratio_df.reset_index()
ratio_df['repertoire_group_name'] = [repertoire_groups[rgid].get('repertoire_group_name') for rgid in ratio_df.loc[:,'repertoire_group_id']]
ratio_df = ratio_df.loc[:,ratio_df.columns[0:3].to_list()+[ratio_df.columns[-1]]+ratio_df.columns[3:-1].to_list()]
ratio_df['repertoire_group_name'] = pd.Categorical(ratio_df['repertoire_group_name'], categories=hue_order, ordered=True) # to put into hue_order
ratio_df = ratio_df.sort_values('repertoire_group_name')

freq_df = group_muts.loc[:,freq_cols]
freq_df['rearrangement'] = freq_df.index.to_series().str.extract(r'-(\d+)$')[0].fillna('All').astype(str)
# freq_df['parent_group'] = [repertoire_groups[freq_df.index[i]].get('repertoire_group_name') if i < 5 else ' '.join(repertoire_groups[freq_df.index[i]].get('repertoire_group_name').split(' ')[0:-1]) for i in range(freq_df.shape[0])]
freq_df = freq_df.loc[:,freq_df.columns[-2:].to_list()+freq_df.columns[:-2].to_list()]
freq_df = freq_df.reset_index()
freq_df['repertoire_group_name'] = [repertoire_groups[rgid].get('repertoire_group_name') for rgid in freq_df.loc[:,'repertoire_group_id']]
freq_df = freq_df.loc[:,freq_df.columns[0:3].to_list()+[freq_df.columns[-1]]+freq_df.columns[3:-1].to_list()]
freq_df['repertoire_group_name'] = pd.Categorical(freq_df['repertoire_group_name'], categories=hue_order, ordered=True) # to put into hue_order
freq_df = freq_df.sort_values('repertoire_group_name')

groups_tuple_repnum_map = {e[1]:len(e[2]) for e in groups_tuple}

## Heatmaps

### Mutation frequency in FWR

In [ ]:
if groups_of_interest:
    idx = groups_of_interest
else:
    idx = list(freq_df['repertoire_group_name'].unique())

value_cols = ['mu_freq_fwr_aa_avg', 'mu_freq_fwr_r_aa_avg', 'mu_freq_fwr_s_aa_avg']
# sns.heatmap(subset)

fig, axes = plt.subplots(3, 1, figsize=(12,24))

for i, value in enumerate(value_cols):
    matrix = freq_df.pivot(
        index="repertoire_group_name",
        columns="rearrangement",
        values=value
    ).reindex(idx)

    # keep only one control group
    # one_control_mat = matrix.drop(matrix.index[0], axis=0)
    one_control_mat = matrix

    # make normalized matrix
    norm_mat = gr.norm_cols(one_control_mat, 'difference')

    # make annotation matrix
    annot_mat = gr.make_annotation_mat(one_control_mat, norm_mat, 4, 4)

    sns.heatmap(
        norm_mat,
        annot=annot_mat,
        fmt='',
        cmap="RdBu_r",
        vmin=-0.08,
        vmax=0.08,
        ax=axes[i]
    )

    yticklabels = [f'{e.get_text()}\n(N = {groups_tuple_repnum_map[e.get_text()]})' for e in  axes[i].get_yticklabels()]
    axes[i].set_yticklabels(yticklabels)
    axes[i].set_title(value)

fig.show()
# ax.set_title('(A)', x=-0.4, fontsize=14, weight='bold')
# ax.set_ylabel('Repertoire Group')
# ax.set_xlabel('Gene Family')

### Mutation frequency in CDR

In [ ]:
if groups_of_interest:
    idx = groups_of_interest
else:
    idx = list(freq_df['repertoire_group_name'].unique())

value_cols = ['mu_freq_cdr_aa_avg', 'mu_freq_cdr_r_aa_avg', 'mu_freq_cdr_s_aa_avg']
# sns.heatmap(subset)

fig, axes = plt.subplots(3, 1, figsize=(12,24))

for i, value in enumerate(value_cols):
    matrix = freq_df.pivot(
        index="repertoire_group_name",
        columns="rearrangement",
        values=value
    ).reindex(idx)

    # keep only one control group
    # one_control_mat = matrix.drop(matrix.index[0], axis=0)
    one_control_mat = matrix

    # make normalized matrix
    norm_mat = gr.norm_cols(one_control_mat, 'difference')

    # make annotation matrix
    annot_mat = gr.make_annotation_mat(one_control_mat, norm_mat, 4, 4)

    sns.heatmap(
        norm_mat,
        annot=annot_mat,
        fmt='',
        cmap="RdBu_r",
        vmin=-0.17,
        vmax=0.17,
        ax=axes[i]
    )

    yticklabels = [f'{e.get_text()}\n(N = {groups_tuple_repnum_map[e.get_text()]})' for e in  axes[i].get_yticklabels()]
    axes[i].set_yticklabels(yticklabels)
    axes[i].set_title(value)

fig.show()
# ax.set_title('(A)', x=-0.4, fontsize=14, weight='bold')
# ax.set_ylabel('Repertoire Group')
# ax.set_xlabel('Gene Family')

### Mutation ratio in FWR

In [ ]:
if groups_of_interest:
    idx = groups_of_interest
else:
    idx = list(freq_df['repertoire_group_name'].unique())

value_cols = ['mu_ratio_fwr_aa_avg', 'mu_ratio_fwr_r_aa_avg', 'mu_ratio_fwr_s_aa_avg']
# sns.heatmap(subset)

fig, axes = plt.subplots(3, 1, figsize=(12,24))

for i, value in enumerate(value_cols):
    matrix = ratio_df.pivot(
        index="repertoire_group_name",
        columns="rearrangement",
        values=value
    ).reindex(idx)

    # keep only one control group
    # one_control_mat = matrix.drop(matrix.index[1], axis=0)
    one_control_mat = matrix

    # make normalized matrix
    norm_mat = gr.norm_cols(one_control_mat, 'difference')

    # make annotation matrix
    annot_mat = gr.make_annotation_mat(one_control_mat, norm_mat, 4, 4)

    sns.heatmap(
        norm_mat,
        annot=annot_mat,
        fmt='',
        cmap="RdBu_r",
        vmin=-0.1,
        vmax=0.1,
        ax=axes[i]
    )

    yticklabels = [f'{e.get_text()}\n(N = {groups_tuple_repnum_map[e.get_text()]})' for e in  axes[i].get_yticklabels()]
    axes[i].set_yticklabels(yticklabels)
    axes[i].set_title(value)

fig.show()
# ax.set_title('(A)', x=-0.4, fontsize=14, weight='bold')
# ax.set_ylabel('Repertoire Group')
# ax.set_xlabel('Gene Family')

### Mutation ratio in CDR

In [ ]:
if groups_of_interest:
    idx = groups_of_interest
else:
    idx = list(freq_df['repertoire_group_name'].unique())

value_cols = ['mu_ratio_cdr_aa_avg', 'mu_ratio_cdr_r_aa_avg', 'mu_ratio_cdr_s_aa_avg']
# sns.heatmap(subset)

fig, axes = plt.subplots(3, 1, figsize=(12,24))

for i, value in enumerate(value_cols):
    matrix = ratio_df.pivot(
        index="repertoire_group_name",
        columns="rearrangement",
        values=value
    ).reindex(idx)

    # keep only one control group
    one_control_mat = matrix.drop(matrix.index[1], axis=0)

    # make normalized matrix
    norm_mat = gr.norm_cols(one_control_mat, 'difference')

    # make annotation matrix
    annot_mat = gr.make_annotation_mat(one_control_mat, norm_mat, 4, 4)

    sns.heatmap(
        norm_mat,
        annot=annot_mat,
        fmt='',
        cmap="RdBu_r",
        vmin=-0.1,
        vmax=0.1,
        ax=axes[i]
    )

    yticklabels = [f'{e.get_text()}\n(N = {groups_tuple_repnum_map[e.get_text()]})' for e in  axes[i].get_yticklabels()]
    axes[i].set_yticklabels(yticklabels)
    axes[i].set_title(value)

fig.show()
# ax.set_title('(A)', x=-0.4, fontsize=14, weight='bold')
# ax.set_ylabel('Repertoire Group')
# ax.set_xlabel('Gene Family')

## Bar graphs

### Mutation ratio in FWR

In [ ]:
gr.plot_ratio(
    ratio_df,
    region='mu_ratio_fwr',
    errorbar='se',
    ylim=(0.6,0.65),
    title='mu_ratio_fwr',
    palette=cond_cmap
)

### Mutation ratio in CDR

In [ ]:
gr.plot_ratio(
    ratio_df,
    region='mu_ratio_cdr',
    errorbar='se',
    ylim=(0.35,0.4),
    title='mu_ratio_cdr',
    palette=cond_cmap
)

### Mutation RS ratio in FWR

In [ ]:
gr.plot_ratio(
    ratio_df,
    region='mu_rsratio_fwr',
    errorbar='se',
    ylim=(1.4,1.6),
    title='mu_rsratio_fwr',
    palette=cond_cmap
)

### Mutation RS ratio in CDR

In [ ]:
gr.plot_ratio(
    ratio_df,
    region='mu_rsratio_cdr',
    errorbar='se',
    ylim=(3.0, 3.4),
    title='mu_rsratio_cdr',
    palette=cond_cmap
)

## Tables

### Mutation RS ratio in FWR

In [ ]:
if groups_of_interest:
    idx = groups_of_interest
else:
    idx = list(freq_df['repertoire_group_name'].unique())

# means to view data easily (temp)
matrix = ratio_df.pivot_table(
    index="repertoire_group_name",
    columns="rearrangement",
    values="mu_rsratio_fwr_aa_avg"
).reindex(idx)
matrix

print(tabulate(matrix, tablefmt='heavy_grid', headers=matrix.columns))

### Mutation RS ratio in CDR

In [ ]:
if groups_of_interest:
    idx = groups_of_interest
else:
    idx = list(freq_df['repertoire_group_name'].unique())

matrix = ratio_df.pivot_table(
    index="repertoire_group_name",
    columns="rearrangement",
    values="mu_rsratio_cdr_aa_avg"
).reindex(idx)
matrix

print(tabulate(matrix, tablefmt='heavy_grid', headers=matrix.columns))

### Mutation ratio

In [ ]:
cols = ['mu_ratio_fwr_aa', 'mu_ratio_cdr_aa', 'mu_ratio_fwr_r_aa', 'mu_ratio_fwr_s_aa', 'mu_ratio_cdr_r_aa', 'mu_ratio_cdr_s_aa', 'mu_rsratio_fwr_aa', 'mu_rsratio_cdr_aa']
true_cols = [[col+'_avg', col+'_std', col+'_N'] for col in cols]
true_cols = [col for true_col in true_cols for col in true_col]

true_cols
df = ratio_df.loc[:,['repertoire_group_name']+true_cols]
df = df.set_index('repertoire_group_name')
if groups_of_interest:
    df = df.reindex(groups_of_interest)

compressed_df = pd.DataFrame(index=df.index, columns=cols)

for col in cols:
    compressed_df.loc[:,col] = df.loc[:,col+'_avg'].astype(float).round(4).astype(str) + \
        ' ± ' + df.loc[:,col+'_std'].astype(float).round(6).astype(str) + \
        '\n' + '(N=' + df.loc[:,col+'_N'].astype(str) + ')'

print(compressed_df.to_markdown())
compressed_df.to_excel('mu_ratio_table.xlsx')

### Mutation frequency

In [ ]:
cols = ['mu_freq_fwr_aa', 'mu_freq_cdr_aa', 'mu_freq_fwr_r_aa', 'mu_freq_fwr_s_aa', 'mu_freq_cdr_r_aa', 'mu_freq_cdr_s_aa']
true_cols = [[col+'_avg', col+'_std', col+'_N'] for col in cols]
true_cols = [col for true_col in true_cols for col in true_col]

df = freq_df.loc[:,['repertoire_group_name']+true_cols]
df = df.set_index('repertoire_group_name')
if groups_of_interest:
    df = df.reindex(groups_of_interest)

compressed_df = pd.DataFrame(index=df.index, columns=cols)

for col in cols:
    compressed_df.loc[:,col] = df.loc[:,col+'_avg'].astype(float).round(4).astype(str) + \
        ' ± ' + df.loc[:,col+'_std'].astype(float).round(6).astype(str) + \
        '\n' + '(N=' + df.loc[:,col+'_N'].astype(str) + ')'

# print(compressed_df.to_markdown())
print(tabulate(compressed_df, tablefmt='heavy_grid', headers=['Group Name']+cols))
compressed_df.to_excel('mu_freq_table.xlsx')

### Mutation ratio and RS ratio

In [ ]:
cols = ['mu_ratio_fwr_aa', 'mu_ratio_cdr_aa', 'mu_ratio_fwr_r_aa', 'mu_ratio_fwr_s_aa', 'mu_ratio_cdr_r_aa', 'mu_ratio_cdr_s_aa']
out_ratio_df = ratio_df.loc[:,['repertoire_group_name']+[col+'_avg' for col in cols]]
out_ratio_df = out_ratio_df.set_index('repertoire_group_name')
out_ratio_df = out_ratio_df.rename(columns={col:'_'.join(col.split('_')[:-1]) for col in out_ratio_df.columns})
temp_ratio_df = ratio_df.set_index('repertoire_group_name')

for col in out_ratio_df.columns:
    out_ratio_df[col] = out_ratio_df[col].round(5).astype(str) + ' ± ' + temp_ratio_df[col+'_std'].round(5).astype(str) +'\n(N='+temp_ratio_df[col+'_N'].astype(str)+')'

print(tabulate(out_ratio_df, tablefmt='heavy_grid', headers=['Group Name']+cols))